<small>Reconocimiento de Patrones · Licenciatura en Ciencias de la Computación · Universidad de Sonora<br>
Julio Waissman · <a href="https://ml-unison.github.io">ml-unison.github.io</a> · Licencia CC BY-SA 4.0</small>

---

# Laboratorio 4 · Regresión logística y Bayes ingenuo desde cero

**Unidad 4**

En clase vimos dos maneras muy distintas de construir un clasificador probabilístico. La **regresión logística** es *discriminativa*: modela directamente $P(y \mid \mathbf{x})$ y ajusta los pesos minimizando la entropía cruzada. **Bayes ingenuo** es *generativo*: modela cómo se producen los datos de cada clase, $P(\mathbf{x} \mid y)$, y llega a $P(y \mid \mathbf{x})$ con la regla de Bayes. En este laboratorio vas a programar los dos: la logística con el **método de Newton** (que, escrito de cierta forma, es el algoritmo de mínimos cuadrados ponderados iterativos, IRLS) y Bayes ingenuo multinomial para clasificar mensajes de texto, trabajando con matrices dispersas y con logaritmos para no perder precisión.

**Cómo se trabaja.** Completa las celdas marcadas con `...`. Después de cada pregunta hay una celda `grader.check("p1")` que verifica tu respuesta: si algo falla, te dice qué. Las preguntas marcadas como *reflexión* se responden por escrito en la celda indicada.

**Reglas.** Dentro de tus funciones usa solo numpy y scipy (`scipy.special.logsumexp` está permitido): nada de `sklearn`, que aquí se usa únicamente para comparar. Las operaciones deben ser vectorizadas: no recorras los datos con ciclos. El ciclo sobre las iteraciones del método de Newton sí está permitido.

**Cómo se entrega.** Ejecuta la libreta de principio a fin (*Entorno de ejecución → Reiniciar y ejecutar todo*), revisa que no haya errores y descarga el archivo `.ipynb`.

Las verificaciones de esta libreta son solo una parte de la revisión: al calificar se usan pruebas adicionales.

In [ ]:
import os, sys

RAMA = "main"  # rama del repositorio del curso de la que se toman paquete, datos y pruebas
os.environ["RDP_RAMA"] = RAMA
if "google.colab" in sys.modules:
    %pip install -q otter-grader "rdp @ git+https://github.com/ml-unison/ml-unison.github.io@{RAMA}#subdirectory=paquete"

In [ ]:
# Verificador de respuestas (otter-grader). Usa las pruebas públicas del laboratorio:
# la copia local si existe; si no (en Colab), las descarga del repositorio del curso.
import pathlib, urllib.request
import otter

PRUEBAS = pathlib.Path("tests") / "u04-lab04-logistica-bayes"
if not PRUEBAS.exists():
    PRUEBAS.mkdir(parents=True)
    for _p in ['p1.py', 'p2.py', 'p3.py', 'p4.py', 'p5.py', 'p6.py', 'p7.py']:
        urllib.request.urlretrieve(
            f"https://raw.githubusercontent.com/ml-unison/ml-unison.github.io/main/unidades/04-lineales/laboratorios/tests/u04-lab04-logistica-bayes/{_p}",
            PRUEBAS / _p)
grader = otter.Notebook(tests_dir=str(PRUEBAS))

In [ ]:
import time
import tracemalloc

import numpy as np
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.special import logsumexp

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

import rdp

rdp.estilo.usar()

# Parte A · Regresión logística con el método de Newton

## Los datos y el problema

Volvemos a los datos de **incumplimiento de pago de tarjetas de crédito** de un banco de Taiwán que usamos en el laboratorio 3 (I.-C. Yeh, *Default of Credit Card Clients*, repositorio UCI, 2009, licencia CC BY 4.0). Cada fila es un cliente; `incumple` vale 1 si el cliente **no pagó** en octubre de 2005. Esta vez usaremos los 30,000 clientes, pero solo cinco variables, para que los pesos se puedan leer uno por uno:

- `limite`: línea de crédito (en nuevos dólares taiwaneses);
- `edad`, en años;
- `atraso_1`: situación de pago en septiembre (−1 pagó a tiempo; 1 a 9, meses de atraso);
- `factura_1`: monto del estado de cuenta de septiembre;
- `pago_1`: monto que el cliente pagó en septiembre.

**El problema.** Queremos estimar la probabilidad de que un cliente incumpla y, sobre todo, **entender** cómo cambia esa probabilidad con cada variable. Para eso la regresión logística es ideal, pero en lugar de pedirle los pesos a scikit-learn vamos a calcularlos nosotros.

Las variables se **estandarizan** con la media y la desviación estándar del conjunto de entrenamiento, así que cada peso mide el efecto de aumentar la variable en **una desviación estándar**.

In [ ]:
credito = rdp.datos.cargar("credito_taiwan")
variables = ["limite", "edad", "atraso_1", "factura_1", "pago_1"]
X_todo = credito[variables].to_numpy(dtype=float)
y_todo = credito["incumple"].to_numpy(dtype=float)

X_ent, X_pru, y_ent, y_pru = train_test_split(X_todo, y_todo, test_size=0.25, stratify=y_todo, random_state=0)
media, escala = X_ent.mean(axis=0), X_ent.std(axis=0)
X_ent = (X_ent - media) / escala
X_pru = (X_pru - media) / escala

print(f"entrenamiento: {X_ent.shape} · prueba: {X_pru.shape} · incumplidos: {y_ent.mean():.1%}")

## 1. La sigmoide, sin desbordamientos

La regresión logística modela la probabilidad de la clase positiva como

$$P(y = 1 \mid \mathbf{x}) = \sigma(\mathbf{w}^\top \mathbf{x}), \qquad \sigma(z) = \frac{1}{1 + e^{-z}} ,$$

donde $\mathbf{x}$ incluye un 1 al principio para el término de intersección $w_0$. En esta parte las etiquetas son $y \in \{0, 1\}$ (la convención de scikit-learn y de la distribución de Bernoulli), no $\{-1, +1\}$ como en *Learning From Data*.

Programar la fórmula tal cual tiene un problema: si $z = -1000$, $e^{-z} = e^{1000}$ no cabe en un número de punto flotante (el máximo es cerca de $e^{709}$) y numpy avisa de un desbordamiento (*overflow*). El resultado final, $1/\infty = 0$, por suerte es correcto, pero en otras fórmulas ese $\infty$ se convierte en `nan`. La salida es usar dos expresiones equivalentes:

$$\sigma(z) = \frac{1}{1 + e^{-z}} \quad (z \ge 0), \qquad \sigma(z) = \frac{e^{z}}{1 + e^{z}} \quad (z < 0).$$

En cada caso el exponente es $-|z| \le 0$, así que la exponencial está entre 0 y 1 y nunca se desborda.

**Pregunta 1.** Escribe `sigmoide(z)` para un arreglo `z` de cualquier forma, usando las dos expresiones de arriba. No debe producir **ningún** aviso de desbordamiento, ni siquiera con $|z| = 1000$. Cuidado: `np.where(cond, a, b)` calcula `a` y `b` completos antes de elegir, así que no basta con escribir las dos fórmulas dentro de un `np.where`. Pista: calcula primero $e^{-|z|}$.

In [ ]:
def sigmoide(z):
    """Función logística, numéricamente estable."""
    ...


sigmoide(np.array([-1000.0, -2.0, 0.0, 2.0, 1000.0]))

In [ ]:
grader.check("p1")

## 2. La entropía cruzada y su gradiente

Sea $X$ la matriz de $N \times (d+1)$ cuyas filas son los $\mathbf{x}_n$ (con el 1 al principio), y $p_n = \sigma(\mathbf{w}^\top \mathbf{x}_n)$. La **verosimilitud** de los datos es $\prod_n p_n^{y_n}(1 - p_n)^{1 - y_n}$; tomar menos el logaritmo y promediar da la **entropía cruzada**:

$$E_{\text{in}}(\mathbf{w}) = -\frac{1}{N}\sum_{n=1}^{N} \Bigl[y_n \log p_n + (1 - y_n)\log(1 - p_n)\Bigr].$$

Con $z_n = \mathbf{w}^\top \mathbf{x}_n$ se cumple $\log p_n = -\log(1 + e^{-z_n})$ y $\log(1 - p_n) = -\log(1 + e^{z_n})$, y con un poco de álgebra cada sumando queda en una forma más cómoda:

$$E_{\text{in}}(\mathbf{w}) = \frac{1}{N}\sum_{n=1}^{N} \Bigl[\log\bigl(1 + e^{z_n}\bigr) - y_n z_n\Bigr].$$

Esta forma no calcula nunca $\log 0$: si el modelo está muy seguro, $p_n$ se redondea a 0 o a 1 en punto flotante y la fórmula original da $-\infty$ o `nan`. Para $\log(1 + e^{z})$ sin desbordamiento numpy tiene `np.logaddexp(0, z)`.

Derivando, y usando que $\sigma'(z) = \sigma(z)\bigl(1 - \sigma(z)\bigr)$, el gradiente es

$$\nabla E_{\text{in}}(\mathbf{w}) = \frac{1}{N}\sum_{n=1}^{N} \bigl(p_n - y_n\bigr)\,\mathbf{x}_n = \frac{1}{N} X^\top (\mathbf{p} - \mathbf{y}).$$

Es la misma forma que el gradiente de los mínimos cuadrados, $\tfrac{2}{N}X^\top(X\mathbf{w} - \mathbf{y})$: «error de predicción por entrada», promediado.

**Pregunta 2.** Escribe `entropia_cruzada(w, X, y)`, que devuelve $E_{\text{in}}(\mathbf{w})$ como `float`, y `gradiente(w, X, y)`, que devuelve el vector $\nabla E_{\text{in}}(\mathbf{w})$ de longitud igual al número de columnas de `X`. Aquí `X` ya trae la columna de unos (abajo está `con_unos` para agregarla) y `y` tiene ceros y unos. Usa la forma estable de $E_{\text{in}}$ y tu `sigmoide`.

In [ ]:
def con_unos(X):
    """Agrega una columna de unos al principio de X."""
    return np.column_stack([np.ones(len(X)), X])


def entropia_cruzada(w, X, y):
    """Entropía cruzada media de la logística con pesos w (X ya incluye la columna de unos)."""
    ...


def gradiente(w, X, y):
    """Gradiente de la entropía cruzada media."""
    ...


A_ent = con_unos(X_ent)
w_cero = np.zeros(A_ent.shape[1])
print(f"E_in(0) = {entropia_cruzada(w_cero, A_ent, y_ent):.4f}  (log 2 = {np.log(2):.4f})")
gradiente(w_cero, A_ent, y_ent)

In [ ]:
grader.check("p2")

## 3. La hessiana

El descenso por gradiente solo usa la pendiente; el método de Newton usa también la **curvatura**. Derivando otra vez el gradiente,

$$\nabla^2 E_{\text{in}}(\mathbf{w}) = \frac{1}{N}\sum_{n=1}^{N} p_n(1 - p_n)\,\mathbf{x}_n \mathbf{x}_n^\top = \frac{1}{N} X^\top S X, \qquad S = \operatorname{diag}\bigl(p_1(1-p_1), \dots, p_N(1-p_N)\bigr).$$

Como $p_n(1 - p_n) > 0$, para cualquier vector $\mathbf{v}$ se tiene $\mathbf{v}^\top \nabla^2 E_{\text{in}}\, \mathbf{v} = \frac{1}{N}\sum_n p_n(1-p_n)(\mathbf{x}_n^\top \mathbf{v})^2 \ge 0$: la hessiana es semidefinida positiva y $E_{\text{in}}$ es **convexa**. Por eso la logística no tiene mínimos locales que no sean globales.

Formar la matriz $S$ de $N \times N$ es un desperdicio: con $N = 22{,}500$ ocuparía unos 4 GB, casi todos ceros. Multiplicar por una matriz diagonal equivale a escalar los renglones: $S X$ es `s[:, None] * X`.

**Pregunta 3.** Escribe `hessiana(w, X)`, que devuelve la matriz $\frac{1}{N}X^\top S X$ de $(d+1) \times (d+1)$, **sin** formar la matriz diagonal $S$.

In [ ]:
def hessiana(w, X):
    """Hessiana de la entropía cruzada media: Xᵀ S X / N."""
    ...


hessiana(w_cero, A_ent).round(3)

In [ ]:
grader.check("p3")

## 4. El método de Newton (IRLS)

Newton aproxima $E_{\text{in}}$ alrededor del punto actual $\mathbf{w}_t$ por su desarrollo de Taylor de segundo orden,

$$E_{\text{in}}(\mathbf{w}) \approx E_{\text{in}}(\mathbf{w}_t) + \mathbf{g}_t^\top(\mathbf{w} - \mathbf{w}_t) + \tfrac{1}{2}(\mathbf{w} - \mathbf{w}_t)^\top H_t (\mathbf{w} - \mathbf{w}_t),$$

y salta al mínimo de esa parábola, que se obtiene igualando su gradiente a cero:

$$\mathbf{w}_{t+1} = \mathbf{w}_t - H_t^{-1}\mathbf{g}_t .$$

No hay tasa de aprendizaje que elegir. ¿Por qué se le llama **mínimos cuadrados ponderados iterativos**? Sustituyendo $\mathbf{g}_t = \frac1N X^\top(\mathbf{p} - \mathbf{y})$ y $H_t = \frac1N X^\top S X$ y reacomodando,

$$\mathbf{w}_{t+1} = (X^\top S X)^{-1} X^\top S\, \mathbf{z}, \qquad \mathbf{z} = X\mathbf{w}_t + S^{-1}(\mathbf{y} - \mathbf{p}),$$

que es la solución de mínimos cuadrados **ponderados** (con pesos $p_n(1-p_n)$) de una respuesta ajustada $\mathbf{z}$. Cada iteración resuelve un problema de mínimos cuadrados; los pesos cambian de una iteración a la siguiente.

También programaremos la versión con **regularización L2**, que minimiza

$$E_{\text{aug}}(\mathbf{w}) = E_{\text{in}}(\mathbf{w}) + \frac{\lambda}{2}\sum_{j=1}^{d} w_j^2 ,$$

sin penalizar la intersección $w_0$ (igual que en *ridge*, en el laboratorio 3). El gradiente gana el término $\lambda\,(0, w_1, \dots, w_d)$ y la hessiana gana $\lambda\,\operatorname{diag}(0, 1, \dots, 1)$. scikit-learn escribe la misma penalización de otra forma: minimiza $C\sum_n(\cdots) + \frac12\lVert \mathbf{w} \rVert^2$, así que $C = 1/(N\lambda)$.

**Pregunta 4.** Escribe `newton_logistica(X, y, lam=0.0, max_iter=25, tol=1e-10)`, que devuelve la tupla `(w0, w)`: la intersección (un `float`) y el vector de $d$ pesos. `X` es de $N \times d$, **sin** la columna de unos: agrégala tú. Empieza en $\mathbf{w} = \mathbf{0}$, haz a lo más `max_iter` pasos de Newton (con `max_iter=0` devuelve ceros) y detente antes si la norma del paso es menor que `tol`. Usa tus `gradiente` y `hessiana` y resuelve el sistema con `np.linalg.solve`, sin invertir la matriz.

In [ ]:
def newton_logistica(X, y, lam=0.0, max_iter=25, tol=1e-10):
    """Regresión logística (con L2 opcional, sin penalizar w0) por el método de Newton. Devuelve (w0, w)."""
    ...


inicio = time.perf_counter()
w0, w = newton_logistica(X_ent, y_ent)
print(f"tiempo: {time.perf_counter() - inicio:.3f} s · w0 = {w0:.4f}")
for nombre, peso in zip(variables, w):
    print(f"  {nombre:>10}: {peso:+.4f}")

In [ ]:
grader.check("p4")

¿Qué tan rápido converge? Abajo corremos Newton con `max_iter` = 0, 1, 2, … y medimos la distancia a la solución final. La escala vertical es logarítmica: en las últimas iteraciones el número de cifras correctas **se duplica** en cada paso, la **convergencia cuadrática** del método de Newton. El descenso por gradiente necesitaría cientos de iteraciones y una tasa de aprendizaje bien elegida para llegar al mismo punto.

In [ ]:
w0_final, w_final = newton_logistica(X_ent, y_ent)
w_final = np.r_[w0_final, w_final]
distancias = []
for k in range(8):
    w0_k, w_k = newton_logistica(X_ent, y_ent, max_iter=k)
    distancias.append(np.linalg.norm(np.r_[w0_k, w_k] - w_final))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.semilogy(range(8), np.maximum(distancias, 1e-16), color=rdp.estilo.AZUL, marker="o")
ax.set_title("Convergencia del método de Newton")
ax.set_xlabel("iteración")
ax.set_ylabel("distancia a la solución (escala log)");

Con los pesos en la mano podemos leer el modelo. Como las variables están estandarizadas, $e^{w_j}$ es la **razón de momios** (*odds ratio*) al aumentar la variable $j$ en una desviación estándar, con las demás fijas: el factor por el que se multiplican los momios $p/(1-p)$ de incumplir.

In [ ]:
print(f"{'variable':>10} {'peso':>8} {'e^peso':>8} {'1 desv. est.':>14}")
for nombre, peso, s in zip(variables, w, escala):
    print(f"{nombre:>10} {peso:+8.3f} {np.exp(peso):8.3f} {s:14,.2f}")
prob_pru = sigmoide(w0 + X_pru @ w)
print(f"\nexactitud en prueba (umbral 0.5): {np.mean((prob_pru >= 0.5) == y_pru):.4f}")

# Parte B · Bayes ingenuo multinomial para mensajes de texto

## Los datos y el problema

La *SMS Spam Collection* (T. A. Almeida, J. M. Gómez Hidalgo y A. Yamakami, 2011; repositorio UCI, licencia CC BY 4.0) reúne 5,574 mensajes de texto en inglés, etiquetados a mano: `spam` = 1 si el mensaje es publicidad no solicitada (13.4 % de ellos) y 0 si es un mensaje normal.

**El problema.** Queremos un filtro que, a partir solo del texto, decida si un mensaje nuevo es *spam*. Es el ejemplo clásico de Bayes ingenuo: los primeros filtros de correo basura de finales de los noventa funcionaban así.

Cada mensaje se representa como una **bolsa de palabras**: un vector de conteos con una entrada por palabra del vocabulario. `CountVectorizer` construye el vocabulario con los mensajes de entrenamiento (7,515 palabras) y devuelve una matriz **dispersa** de scipy: casi todas las entradas son cero, porque cada mensaje usa unas cuantas palabras de miles posibles, y la matriz solo guarda las que no lo son.

In [ ]:
mensajes = rdp.datos.cargar("sms_spam")
m_ent, m_pru = train_test_split(mensajes, test_size=0.25, stratify=mensajes.spam, random_state=2011)
vectorizador = CountVectorizer().fit(m_ent.texto)
C_ent = vectorizador.transform(m_ent.texto)
C_pru = vectorizador.transform(m_pru.texto)
t_ent = m_ent.spam.to_numpy()
t_pru = m_pru.spam.to_numpy()
vocabulario = vectorizador.get_feature_names_out()

print(f"entrenamiento: {C_ent.shape} · prueba: {C_pru.shape} · tipo: {type(C_ent).__name__}")
print(f"entradas distintas de cero: {C_ent.nnz:,} de {np.prod(C_ent.shape):,} ({C_ent.nnz / np.prod(C_ent.shape):.2%})")
print(m_ent.texto.iloc[0], "→", {vocabulario[j]: int(c) for j, c in zip(C_ent[0].indices, C_ent[0].data)})

## 5. Ajustar el modelo

Bayes ingenuo multinomial imagina que un mensaje de la clase $c$ se escribe sacando palabras, una tras otra y **de manera independiente**, de una urna con probabilidades $\theta_{c1}, \dots, \theta_{cV}$ (una por palabra del vocabulario). Si el mensaje $\mathbf{x}$ tiene $x_j$ veces la palabra $j$,

$$P(\mathbf{x} \mid y = c) \;\propto\; \prod_{j=1}^{V} \theta_{cj}^{\,x_j}.$$

La suposición es claramente falsa («free» y «prize» aparecen juntas mucho más de lo que sugiere su frecuencia por separado), pero el clasificador funciona sorprendentemente bien.

Los estimadores de máxima verosimilitud son frecuencias: la previa $\pi_c = N_c/N$ es la fracción de mensajes de la clase $c$, y $\theta_{cj} = T_{cj} / T_c$, donde $T_{cj}$ es cuántas veces aparece la palabra $j$ en **todos** los mensajes de la clase $c$ y $T_c = \sum_j T_{cj}$. Con el **suavizado de Laplace** de parámetro $\alpha > 0$ sumamos $\alpha$ «apariciones ficticias» de cada palabra:

$$\hat\theta_{cj} = \frac{T_{cj} + \alpha}{T_c + \alpha V}.$$

El $\alpha V$ del denominador es lo que hace que $\sum_j \hat\theta_{cj} = 1$. Guardaremos todo en **logaritmos**, porque es lo que vamos a sumar en la pregunta 6.

**Pregunta 5.** Escribe `ajustar_nb(X, y, alpha=1.0)`, que recibe una matriz dispersa `X` de conteos ($N \times V$) y etiquetas `y` con valores 0 y 1, y devuelve la tupla `(log_previas, log_theta)`:

- `log_previas`: arreglo de longitud 2 con $\log \pi_0$, $\log \pi_1$;
- `log_theta`: arreglo de $2 \times V$ cuyo renglón $c$ tiene los $\log \hat\theta_{cj}$.

**No densifiques** `X` (nada de `X.toarray()` ni `X.todense()`): con un vocabulario grande la matriz densa no cabe en memoria. Las matrices dispersas de scipy saben sumar por columnas (`X[filas].sum(axis=0)`), y el resultado se convierte en un arreglo plano con `np.asarray(...).ravel()`.

In [ ]:
def ajustar_nb(X, y, alpha=1.0):
    """Bayes ingenuo multinomial con suavizado de Laplace. Devuelve (log_previas, log_theta)."""
    ...


log_previas, log_theta = ajustar_nb(C_ent, t_ent, alpha=1.0)
print("previas:", np.exp(log_previas).round(4), "· forma de log_theta:", log_theta.shape)

In [ ]:
grader.check("p5")

Antes de clasificar, veamos qué aprendió el modelo. El cociente $\log(\hat\theta_{1j}/\hat\theta_{0j})$ dice cuánto más frecuente es la palabra $j$ en el *spam* que en los mensajes normales. Las palabras con el cociente más alto y más bajo:

In [ ]:
cociente = log_theta[1] - log_theta[0]
orden = np.argsort(cociente)
print("más típicas del spam:  ", ", ".join(vocabulario[orden[-15:][::-1]]))
print("más típicas de lo normal:", ", ".join(vocabulario[orden[:15]]))

## 6. La probabilidad posterior, con logaritmos

Por la regla de Bayes, $P(y = c \mid \mathbf{x}) \propto \pi_c \prod_j \theta_{cj}^{x_j}$. El producto de cientos de probabilidades pequeñas se vuelve exactamente 0 en punto flotante (*subdesbordamiento*), así que trabajamos con la **log-probabilidad conjunta**

$$J_c(\mathbf{x}) = \log \pi_c + \sum_{j=1}^{V} x_j \log \hat\theta_{cj},$$

que para todos los mensajes a la vez es un producto de matrices: $J = X\,(\log\hat\theta)^\top + \log\boldsymbol\pi$, de $N \times 2$. Para clasificar basta comparar $J_0$ con $J_1$. Para obtener probabilidades hay que normalizar:

$$P(y = c \mid \mathbf{x}) = \frac{e^{J_c}}{e^{J_0} + e^{J_1}} = \exp\Bigl(J_c - \log\bigl(e^{J_0} + e^{J_1}\bigr)\Bigr).$$

Calcular $e^{J_c}$ directamente subdesborda en mensajes largos ($J_c$ puede valer −3,000). El truco **log-suma-exp** resta el máximo antes de exponenciar: $\log\sum_c e^{J_c} = m + \log\sum_c e^{J_c - m}$ con $m = \max_c J_c$. Lo hace `scipy.special.logsumexp`.

**Pregunta 6.** Escribe `log_conjunta(X, log_previas, log_theta)`, que devuelve el arreglo denso $J$ de $N \times 2$, y `proba_nb(X, log_previas, log_theta)`, que devuelve el arreglo de $N \times 2$ de probabilidades posteriores (cada renglón suma 1), calculado con log-suma-exp. `X` es dispersa; el resultado de multiplicarla por un arreglo denso ya es denso.

In [ ]:
def log_conjunta(X, log_previas, log_theta):
    """Log-probabilidad conjunta log π_c + Σ_j x_j log θ_cj, de N × 2."""
    ...


def proba_nb(X, log_previas, log_theta):
    """Probabilidades posteriores P(y = c | x), de N × 2, calculadas de forma estable."""
    ...


proba_nb(C_pru[:5], log_previas, log_theta).round(4)

In [ ]:
grader.check("p6")

## 7. ¿Qué tan bien filtra?

**Pregunta 7.** Con tus funciones y $\alpha = 1$, ajustadas con `C_ent` y `t_ent`, calcula para el conjunto de **prueba**:

- `prob_spam`: arreglo con la probabilidad posterior de *spam* de cada mensaje;
- `pred_nb`: la clase predicha (0 o 1), la más probable;
- `exactitud_nb`: la fracción de aciertos (un `float`).

In [ ]:
prob_spam = ...
pred_nb = ...
exactitud_nb = ...
print(f"exactitud en prueba: {exactitud_nb:.4f}")

In [ ]:
grader.check("p7")

## 8. ¿Generativo o discriminativo?

A. Ng y M. Jordan (*On discriminative vs. generative classifiers*, NIPS 2001) compararon Bayes ingenuo con la regresión logística, que forman un «par»: con las mismas variables, las dos producen fronteras de decisión lineales (el $\log$ del cociente de posteriores de Bayes ingenuo es lineal en $\mathbf{x}$). Su conclusión: el modelo **generativo** se acerca a su error asintótico mucho más rápido (con un número de ejemplos del orden de $\log d$ en vez de $d$), mientras que el **discriminativo** suele tener un error asintótico menor. Es decir, con pocos datos se espera que gane Bayes ingenuo y, con muchos, la logística.

La celda siguiente construye una **curva de aprendizaje**: para cada tamaño $n$ toma 20 submuestras al azar del conjunto de entrenamiento (sin reemplazo; se descartan las que tienen una sola clase), reajusta el vocabulario con la submuestra, entrena tu Bayes ingenuo y una `LogisticRegression(C=10)` de scikit-learn, y mide el error en el mismo conjunto de prueba. Tarda menos de un minuto.

In [ ]:
tamanos = [20, 50, 100, 200, 500, 1000, 2000, 4180]
rng = np.random.default_rng(0)
error_nb, error_log = [], []
for n in tamanos:
    e_nb, e_log = [], []
    for repeticion in range(20 if n < len(m_ent) else 1):
        indices = rng.choice(len(m_ent), n, replace=False) if n < len(m_ent) else np.arange(len(m_ent))
        sub = m_ent.iloc[indices]
        if sub.spam.nunique() < 2:
            continue
        vec = CountVectorizer().fit(sub.texto)
        A, B = vec.transform(sub.texto), vec.transform(m_pru.texto)
        lp, lt = ajustar_nb(A, sub.spam.to_numpy(), alpha=1.0)
        e_nb.append(np.mean(np.argmax(log_conjunta(B, lp, lt), axis=1) != t_pru))
        logistica = LogisticRegression(C=10, max_iter=3000).fit(A, sub.spam)
        e_log.append(np.mean(logistica.predict(B) != t_pru))
    error_nb.append(np.mean(e_nb))
    error_log.append(np.mean(e_log))
    print(f"n = {n:>5}: Bayes ingenuo {error_nb[-1]:.3f} · logística {error_log[-1]:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(tamanos, error_nb, color=rdp.estilo.AZUL, marker="o")
ax.plot(tamanos, error_log, color=rdp.estilo.ROJO, marker="o")
ax.set_xscale("log")
ax.set_xlim(15, 9000)
ax.set_xticks(tamanos, [f"{n:,}" for n in tamanos])
ax.minorticks_off()
rdp.estilo.etiquetar(ax, 4600, {"Bayes ingenuo": error_nb[-1], "logística": error_log[-1]})
ax.set_title("Curva de aprendizaje en los SMS: error de prueba contra tamaño de entrenamiento")
ax.set_xlabel("mensajes de entrenamiento (escala log)")
ax.set_ylabel("error de prueba");

## 9. Reflexión

Estas preguntas no se verifican automáticamente: respóndelas en la celda de texto que sigue a cada una. Si necesitas hacer cuentas, agrega celdas de código.

<!-- BEGIN QUESTION -->

**Reflexión 1.** Toma el peso de `atraso_1` que obtuviste con Newton (sin regularización). ¿Cuál es la razón de momios asociada a una desviación estándar más de atraso? Conviértela a la razón de momios por **cada mes adicional** de atraso (usa `escala`) y explica en una o dos frases qué significa para el banco, en términos de momios y no de probabilidad. ¿Por qué el mismo aumento en los momios se traduce en cambios de probabilidad distintos para un cliente con $p = 0.05$ que para uno con $p = 0.5$? ¿Se puede leer ese peso como «el efecto causal» de atrasarse?

_Escribe aquí tu respuesta._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

**Reflexión 2.** Prueba tu `ajustar_nb` con `alpha=1e-10` (casi sin suavizado) y mide la exactitud en prueba. Busca un mensaje de prueba normal que quede mal clasificado y explica qué pasó: ¿qué ocurre con $\hat\theta_{cj}$ cuando una palabra nunca apareció en los mensajes de entrenamiento de la clase $c$, y qué le hace eso a $J_c$? ¿Qué pasaría con $\alpha = 0$ exactamente? Explica por qué el suavizado de Laplace resuelve el problema y por qué el denominador necesita el término $\alpha V$.

_Escribe aquí tu respuesta._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

**Reflexión 3.** Con base en la curva de aprendizaje de la sección 8, ¿se confirma la predicción de Ng y Jordan en estos datos? Describe lo que ves con pocos y con muchos datos, y di con honestidad qué parte de la predicción **no** se puede comprobar aquí y por qué. ¿Qué harías para tener más evidencia? Explica, en términos de cuántos parámetros estima cada modelo y de qué suposiciones hace, por qué el modelo generativo puede aprender más rápido.

_Escribe aquí tu respuesta._

<!-- END QUESTION -->



---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
grader.check_all()